# Checkpoint A2: Chạy test pass đầu tiên

Jupyter Notebook này thực hiện tự động hóa việc khởi tạo thư mục dự án, chuẩn bị tệp tin mã nguồn sườn, sao chép dữ liệu kiểm thử từ Session trước, và chạy thử nghiệm anonymizer để báo cáo tỷ lệ PASS/FAIL ban đầu.

### Bước 1: Tự động khởi tạo cấu trúc mã nguồn sườn `src/anonymizer.py`
Nếu bạn chưa tạo thư mục `src` và file `anonymizer.py`, chạy cell dưới đây sẽ tự động tạo cấu trúc này cùng với phiên bản Regex cơ bản ban đầu để bạn kiểm tra luồng chạy ngay lập tức.

In [1]:
import sys
from pathlib import Path

# Định vị thư mục session hiện tại
current_dir = Path.cwd()
session_dir = None
for parent in [current_dir] + list(current_dir.parents):
    if parent.name == "session-06-compliance-capstone":
        session_dir = parent
        break
if not session_dir:
    session_dir = current_dir

src_dir = session_dir / "src"
src_dir.mkdir(parents=True, exist_ok=True)
code_path = src_dir / "anonymizer.py"

if not code_path.exists():
    print(f"Tạo tệp tin mã nguồn sườn tại: {code_path}")
    skeleton_code = (
        "import re\n"
        "import json\n"
        "from pathlib import Path\n\n"
        "# Regex cho các loại PII chuẩn\n"
        "PATTERNS = {\n"
        "    'email': re.compile(r'\\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}\\b'),\n"
        "    'phone': re.compile(r'\\b(?:\\+84\\s?|0)\\d{2,3}[\\s.-]?\\d{3}[\\s.-]?\\d{3,4}\\b'),\n"
        "    'cccd': re.compile(r'\\b\\d{12}\\b'),\n"
        "}\n\n"
        "def anonymize_text(text: str) -> str:\n"
        "    if not text:\n"
        "        return ''\n"
        "    result = text\n"
        "    # Ẩn Email\n"
        "    result = PATTERNS['email'].sub('[REDACTED_EMAIL]', result)\n"
        "    # Ẩn Số điện thoại\n"
        "    result = PATTERNS['phone'].sub('[REDACTED_PHONE]', result)\n"
        "    # Ẩn CCCD\n"
        "    result = PATTERNS['cccd'].sub('[REDACTED_CCCD]', result)\n"
        "    return result\n"
    )
    code_path.write_text(skeleton_code, encoding="utf-8")
    print("[OK] Đã tạo file src/anonymizer.py thành công. Học viên sẽ nâng cấp logic tại file này.")
else:
    print(f"[OK] Đã tìm thấy tệp mã nguồn bài làm của học viên tại: {code_path.name}")

Tạo tệp tin mã nguồn sườn tại: /Users/shimazu/Documents/9. active/course_viettel_network/vtn-ai-builders-bootcamp-2026/03-practice/day-03/session-06-compliance-capstone/src/anonymizer.py
[OK] Đã tạo file src/anonymizer.py thành công. Học viên sẽ nâng cấp logic tại file này.


### Bước 2: Xác minh dữ liệu mẫu kiểm thử
Chạy cell này để kiểm tra sự hiện diện của các tệp tin dữ liệu mẫu PII trong thư mục `synthetic-data` của Session 06. Nếu không tìm thấy, hệ thống sẽ tự sinh dữ liệu mẫu chuẩn cho bạn.

In [2]:
import shutil

target_data_dir = session_dir / "synthetic-data"
target_data_dir.mkdir(parents=True, exist_ok=True)

files_to_check = ["pii-sample-01.txt", "pii-sample-02-tricky.txt"]

for fname in files_to_check:
    target_file = target_data_dir / fname
    if target_file.exists():
        print(f"[OK] Đã có file kiểm thử: {fname}")
    else:
        print(f"Không tìm thấy file {fname}. Tự động khởi tạo dữ liệu mẫu...")
        if fname == "pii-sample-01.txt":
            content = (
                "TAI LIEU MO PHONG - KHONG PHAI DU LIEU THAT\n\n"
                "Bien ban tiep nhan yeu cau ho tro noi bo\n\n"
                "Nguoi yeu cau: Nguyen Van A\n"
                "So CCCD: 012345678901\n"
                "So dien thoai: 0987654321\n"
                "Email: nguyenvana.demo@example.local\n"
                "Dia chi mo phong: So 12 duong Mo Phong, phuong Du Lieu Mau, thanh pho Demo\n\n"
                "Noi dung yeu cau:\n"
                "Can ho tro kiem tra checklist van hanh truoc khi gui tai lieu sang nhom xu ly tiep theo.\n"
            )
        else:
            content = (
                "Email la viet tat cua Electronic Mail. "
                "Khach hang Tran Van B, email tranb@yahoo.com, "
                "SDT 0987654321, CCCD 079201005678. "
                "Ghi chu: So CCCD 079abc01234xyz khong hop le do chua ky tu la."
            )
        target_file.write_text(content, encoding="utf-8")
        print(f"Đã tạo file dữ liệu mẫu: {fname}")

[OK] Đã có file kiểm thử: pii-sample-01.txt
[OK] Đã có file kiểm thử: pii-sample-02-tricky.txt


### Bước 3: Import module bài làm của học viên
Nạp hàm `anonymize_text` từ file `src/anonymizer.py` của bạn để bắt đầu kiểm thử.

In [3]:
import sys

# Thêm src vào sys.path để import chéo nền tảng
sys.path.insert(0, str(src_dir))

try:
    if "anonymizer" in sys.modules:
        del sys.modules["anonymizer"]
    from anonymizer import anonymize_text
    print("[OK] Import anonymize_text thành công từ src/anonymizer.py!")
except ImportError as e:
    print(f"[LỖI] Không thể import module: {e}")
    raise e

[OK] Import anonymize_text thành công từ src/anonymizer.py!


### Bước 4: Chạy thử trên dữ liệu mẫu
Chạy thử anonymizer trên `pii-sample-01.txt` và xem kết quả đầu ra sau khi che giấu.

In [4]:
sample_1 = (target_data_dir / "pii-sample-01.txt").read_text(encoding="utf-8")
print("=== VĂN BẢN GỐC ===")
print(sample_1)

print("\n=== VĂN BẢN ĐÃ CHE GIẤU (ANONYMIZED) ===")
result_1 = anonymize_text(sample_1)
print(result_1)

=== VĂN BẢN GỐC ===
TAI LIEU MO PHONG - KHONG PHAI DU LIEU THAT

Bien ban tiep nhan yeu cau ho tro noi bo

Nguoi yeu cau: Nguyen Van A
So CCCD: 012345678901
So dien thoai: 0987654321
Email: nguyenvana.demo@example.local
Dia chi mo phong: So 12 duong Mo Phong, phuong Du Lieu Mau, thanh pho Demo

Noi dung yeu cau:
Can ho tro kiem tra checklist van hanh truoc khi gui tai lieu sang nhom xu ly tiep theo.



=== VĂN BẢN ĐÃ CHE GIẤU (ANONYMIZED) ===
TAI LIEU MO PHONG - KHONG PHAI DU LIEU THAT

Bien ban tiep nhan yeu cau ho tro noi bo

Nguoi yeu cau: Nguyen Van A
So CCCD: [REDACTED_CCCD]
So dien thoai: [REDACTED_PHONE]
Email: [REDACTED_EMAIL]
Dia chi mo phong: So 12 duong Mo Phong, phuong Du Lieu Mau, thanh pho Demo

Noi dung yeu cau:
Can ho tro kiem tra checklist van hanh truoc khi gui tai lieu sang nhom xu ly tiep theo.




### Bước 5: Tự động chạy 10 Test Cases & Xuất báo cáo kết quả
Cell dưới đây sẽ tự động nạp 10 ca kiểm thử đã đặc tả ở Checkpoint A1, kiểm tra đầu ra của anonymizer và xuất báo cáo dạng bảng chi tiết.

In [5]:
# Import test_cases từ Checkpoint A1 bằng cách định nghĩa lại hoặc lấy biến
# Để độc lập và tự động, chúng ta định nghĩa lại nhanh tập test_cases
test_cases_local = {
    "TC-01": {"mo_ta": "Email cá nhân trong câu thường", "dau_vao": "Vui lòng gửi hợp đồng đến nguyenvana@gmail.com", "check": "[REDACTED_EMAIL]"},
    "TC-02": {"mo_ta": "Số điện thoại Việt Nam", "dau_vao": "Liên hệ tôi qua số 0912345678 nhé", "check": "[REDACTED_PHONE]"},
    "TC-03": {"mo_ta": "Số CCCD chuẩn", "dau_vao": "Căn cước công dân số 079201001234", "check": "[REDACTED_CCCD]"},
    "TC-04": {"mo_ta": "Email sai định dạng (thiếu @)", "dau_vao": "Gửi cho nguyenvana.gmail.com giúp tôi", "check": "nguyenvana.gmail.com"},
    "TC-05": {"mo_ta": "SĐT quá dài", "dau_vao": "Gọi cho tôi số 0912345678910", "check": "0912345678910"},
    "TC-06": {"mo_ta": "CCCD sai format", "dau_vao": "CCCD số 079abc01234xyz", "check": "079abc01234xyz"},
    "TC-07": {"mo_ta": "Chuỗi rỗng", "dau_vao": "", "check": ""},
    "TC-08": {"mo_ta": "Văn bản không PII", "dau_vao": "Hôm nay thời tiết đẹp quá", "check": "Hôm nay thời tiết đẹp quá"},
    "TC-09": {"mo_ta": "Nhiều PII trong cùng câu", "dau_vao": "KH Trần Văn B, email tranb@yahoo.com, SĐT 0987654321, CCCD 079201005678", "check": "[REDACTED_"},
    "TC-10": {"mo_ta": "Danh từ chung kỹ thuật trùng PII", "dau_vao": "Email là viết tắt của Electronic Mail", "check": "Email"},
}

results = []
pass_count = 0

print(f"{'ID':<5} | {'Mô tả ca test':<35} | {'Kết quả':<8} | {'Chi tiết lỗi (nếu có)'}")
print("-" * 85)

for tid, tc in test_cases_local.items():
    out = anonymize_text(tc["dau_vao"])
    is_pass = False
    err = ""
    
    # Kiểm tra điều kiện pass
    if tid in ["TC-01", "TC-02", "TC-03"]:
        is_pass = tc["check"] in out
        if not is_pass: err = f"Thiếu {tc['check']}"
    elif tid in ["TC-04", "TC-05", "TC-06", "TC-08", "TC-10"]:
        # Không được che nhầm
        is_pass = tc["check"] in out and "[REDACTED_" not in out
        if not is_pass: err = f"Bị ẩn nhầm hoặc mất dữ liệu gốc"
    elif tid == "TC-07":
        is_pass = out == ""
        if not is_pass: err = "Đầu ra không rỗng"
    elif tid == "TC-09":
        # Phải che nhiều trường
        is_pass = out.count("[REDACTED_") >= 3
        if not is_pass: err = f"Số PII được ẩn ít hơn mong đợi ({out.count('[REDACTED_')}/3)"
        
    status = "PASS" if is_pass else "FAIL"
    if is_pass: 
        pass_count += 1
    print(f"{tid:<5} | {tc['mo_ta']:<35} | {status:<8} | {err}")

total = len(test_cases_local)
print("-" * 85)
print(f"Tỷ lệ thành công: {pass_count}/{total} ca ({pass_count/total*100:.1f}%)")

if pass_count == total:
    print("\n=> [TUYỆT VỜI] 10/10 PASS! Mã nguồn Regex của bạn hoạt động rất tốt.")
    print("Bạn đã sẵn sàng bước sang Phần B (Phòng thủ Prompt Injection).")
else:
    print(f"\n=> [CẢNH BÁO] Có {total - pass_count} ca kiểm thử thất bại.")
    print("Học viên cần nâng cấp và tối ưu hóa file src/anonymizer.py để khắc phục lỗi.")

ID    | Mô tả ca test                       | Kết quả  | Chi tiết lỗi (nếu có)
-------------------------------------------------------------------------------------
TC-01 | Email cá nhân trong câu thường      | PASS     | 
TC-02 | Số điện thoại Việt Nam              | PASS     | 
TC-03 | Số CCCD chuẩn                       | PASS     | 
TC-04 | Email sai định dạng (thiếu @)       | PASS     | 
TC-05 | SĐT quá dài                         | PASS     | 
TC-06 | CCCD sai format                     | PASS     | 
TC-07 | Chuỗi rỗng                          | PASS     | 
TC-08 | Văn bản không PII                   | PASS     | 
TC-09 | Nhiều PII trong cùng câu            | PASS     | 
TC-10 | Danh từ chung kỹ thuật trùng PII    | PASS     | 
-------------------------------------------------------------------------------------
Tỷ lệ thành công: 10/10 ca (100.0%)

=> [TUYỆT VỜI] 10/10 PASS! Mã nguồn Regex của bạn hoạt động rất tốt.
Bạn đã sẵn sàng bước sang Phần B (Phòng thủ Prompt Injection).


---
**Hoàn thành Checkpoint A2!** Hãy quay lại file hướng dẫn **[lab.md](../lab.md)** để chuyển sang **Phần B: Compliance Check & Prompt Injection Defense**.